# Fine-tune TrOCR on `sample_1000` (Polish handwriting)

Fine-tunes `microsoft/trocr-base-handwritten` on the 1000 labeled line crops in
`output/sample_1000/` + `output/sample_1000_transcribed_v2.txt`, then evaluates on a held-out split.

**Before running:**
1. `Runtime > Change runtime type` and select a **GPU** (T4 is fine).
2. Upload your data to Google Drive — see Step 1 below.

In [1]:
!pip install -q "transformers>=4.35.0" "datasets>=2.14.0" accelerate

## Step 1 — Get the data into Colab

Upload these two files to your Google Drive (e.g. into `MyDrive/UMwTI/`):

- `sample_1000.zip` — a zip of the `output/sample_1000/` folder (1000 PNG line images)
- `sample_1000_transcribed_v2.txt` — the tab-separated transcription file

To create the zip locally (run once on your machine, not in Colab):

```python
import shutil
shutil.make_archive("sample_1000", "zip", "output", "sample_1000")
```

Then set `DATA_DIR` below to the Drive folder you uploaded the files to and run the cell.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import shutil
import zipfile

DATA_DIR = Path("/content/drive/MyDrive/UMwTI")   # where you uploaded the files

WORK_DIR = Path("/content/data")
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)        # clear stale files from previous runs
WORK_DIR.mkdir(parents=True, exist_ok=True)

IMAGES_DIR = WORK_DIR / "combined_dataset"
TRANSCRIPTIONS_FILE = WORK_DIR / "combined_transcribed.txt"

if not IMAGES_DIR.exists():
    zip_path = DATA_DIR / "combined_dataset.zip"
    print(f"Extracting {zip_path} ...")
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(WORK_DIR)
    if not IMAGES_DIR.exists():
        IMAGES_DIR = WORK_DIR

if not TRANSCRIPTIONS_FILE.exists():
    shutil.copy(DATA_DIR / "combined_transcribed.txt", TRANSCRIPTIONS_FILE)

n_images = len(list(IMAGES_DIR.glob("*.png")))
print("Images dir:", IMAGES_DIR, "-", n_images, "images")
print("Transcriptions file:", TRANSCRIPTIONS_FILE)

KeyboardInterrupt: 

## Step 2 — Config & imports

In [2]:
from __future__ import annotations

import inspect
import logging
import random
from dataclasses import dataclass

import numpy as np
import torch
from datasets import Dataset
from PIL import Image
from transformers import (
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    TrOCRProcessor,
    VisionEncoderDecoderModel,
)

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)-7s | %(message)s",
    datefmt="%H:%M:%S",
)
logger = logging.getLogger(__name__)

TROCR_MODEL_BASE = "microsoft/trocr-base-handwritten"

# Where the fine-tuned model gets saved (on your Drive, so it survives the session)
OUTPUT_MODEL_DIR = Path("/content/drive/MyDrive/UMwTI/models/trocr_sample1000")

# Hyperparameters
EPOCHS = 10
BATCH_SIZE = 12
LEARNING_RATE = 5e-5
TEST_SIZE = 0.15
SEED = 42
MAX_TARGET_LENGTH = 128
PRINT_TEST_EXAMPLES = 10

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

Using device: cuda


## Step 3 — Load samples and build train/test splits

`sample_1000_transcribed_v2.txt` is tab-separated `image_name<TAB>text`.
Lines with an empty transcription (blank/illegible crops) are skipped.

In [3]:
def _levenshtein_distance(a: str, b: str) -> int:
    """Compute Levenshtein distance with dynamic programming."""
    if a == b:
        return 0
    if not a:
        return len(b)
    if not b:
        return len(a)

    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, start=1):
        curr = [i]
        for j, cb in enumerate(b, start=1):
            cost = 0 if ca == cb else 1
            curr.append(min(
                curr[j - 1] + 1,      # insertion
                prev[j] + 1,          # deletion
                prev[j - 1] + cost,   # substitution
            ))
        prev = curr
    return prev[-1]


def _character_error_rate(refs: list[str], preds: list[str]) -> float:
    total_dist = 0
    total_chars = 0
    for ref, pred in zip(refs, preds):
        total_dist += _levenshtein_distance(ref, pred)
        total_chars += max(1, len(ref))
    return total_dist / total_chars


def load_samples(images_dir: Path, transcription_file: Path) -> list[dict[str, str]]:
    """Load labeled samples and validate image existence."""
    if not images_dir.exists():
        raise FileNotFoundError(f"Images directory not found: {images_dir}")
    if not transcription_file.exists():
        raise FileNotFoundError(f"Transcription file not found: {transcription_file}")

    records: list[dict[str, str]] = []
    skipped_empty = 0
    with open(transcription_file, "r", encoding="utf-8") as f:
        for idx, raw_line in enumerate(f, start=1):
            line = raw_line.rstrip("\n")
            if not line.strip():
                continue
            if "\t" not in line:
                raise ValueError(f"Line {idx} in {transcription_file} is not tab-separated.")

            image_name, text = line.split("\t", 1)
            if not text.strip():
                skipped_empty += 1
                continue

            image_path = images_dir / image_name
            if not image_path.exists():
                raise FileNotFoundError(f"Missing image for line {idx}: {image_name}")

            records.append({"image_path": str(image_path), "text": text})

    if not records:
        raise ValueError("No valid samples were loaded.")

    logger.info(
        "Loaded %d labeled samples (%d skipped: empty transcription).",
        len(records), skipped_empty,
    )
    return records


def build_datasets(records: list[dict[str, str]], test_size: float, seed: int) -> tuple[Dataset, Dataset]:
    """Create train/test HF datasets."""
    dataset = Dataset.from_list(records)
    split = dataset.train_test_split(test_size=test_size, seed=seed, shuffle=True)
    train_ds = split["train"]
    test_ds = split["test"]
    logger.info("Split dataset: %d train / %d test", len(train_ds), len(test_ds))
    return train_ds, test_ds


records = load_samples(IMAGES_DIR, TRANSCRIPTIONS_FILE)
train_raw, test_raw = build_datasets(records, test_size=TEST_SIZE, seed=SEED)

## Step 4 — Load model & processor

In [4]:
processor = TrOCRProcessor.from_pretrained(TROCR_MODEL_BASE)
model = VisionEncoderDecoderModel.from_pretrained(TROCR_MODEL_BASE)

# Ensure generation tokens are consistently configured.
model.config.decoder_start_token_id = processor.tokenizer.cls_token_id
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.eos_token_id = processor.tokenizer.sep_token_id
model.config.vocab_size = model.config.decoder.vocab_size

if model.generation_config is not None:
    model.generation_config.decoder_start_token_id = processor.tokenizer.cls_token_id
    model.generation_config.pad_token_id = processor.tokenizer.pad_token_id
    model.generation_config.eos_token_id = processor.tokenizer.sep_token_id

model.to(device)
print("Model loaded on", device)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading weights:   0%|          | 0/478 [00:00<?, ?it/s]

[transformers] VisionEncoderDecoderModel LOAD REPORT from: microsoft/trocr-base-handwritten
Key                         | Status  | 
----------------------------+---------+-
encoder.pooler.dense.bias   | MISSING | 
encoder.pooler.dense.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded on cuda


## Step 5 — Preprocess datasets

In [5]:
def preprocess_dataset(ds: Dataset, processor: TrOCRProcessor, max_target_length: int) -> Dataset:
    """Tokenize images and text for TrOCR training."""

    def _map_fn(examples: dict[str, list[str]]) -> dict[str, list[list[int]]]:
        images = [Image.open(path).convert("RGB") for path in examples["image_path"]]
        pixel_values = processor(images=images, return_tensors="pt").pixel_values

        labels = processor.tokenizer(
            examples["text"],
            padding="max_length",
            max_length=max_target_length,
            truncation=True,
        ).input_ids

        # Ignore pad tokens in the loss
        labels = [
            [token if token != processor.tokenizer.pad_token_id else -100 for token in seq]
            for seq in labels
        ]

        return {
            "pixel_values": pixel_values.tolist(),
            "labels": labels,
        }

    return ds.map(
        _map_fn,
        batched=True,
        batch_size=32,
        remove_columns=ds.column_names,
        desc="Preprocessing",
    )


@dataclass
class TrOCRDataCollator:
    """Batch collator for preprocessed TrOCR tensors."""

    def __call__(self, features: list[dict[str, list[int]]]) -> dict[str, torch.Tensor]:
        pixel_values = torch.tensor([f["pixel_values"] for f in features], dtype=torch.float32)
        labels = torch.tensor([f["labels"] for f in features], dtype=torch.long)
        return {"pixel_values": pixel_values, "labels": labels}


train_ds = preprocess_dataset(train_raw, processor, max_target_length=MAX_TARGET_LENGTH)
test_ds = preprocess_dataset(test_raw, processor, max_target_length=MAX_TARGET_LENGTH)

Preprocessing:   0%|          | 0/833 [00:00<?, ? examples/s]

Preprocessing:   0%|          | 0/147 [00:00<?, ? examples/s]

## Step 6 — Metrics, training args & trainer

In [6]:
def build_compute_metrics(processor: TrOCRProcessor):
    """Build CER + exact-match metrics for Trainer."""

    def _compute_metrics(eval_preds):
        pred_ids, label_ids = eval_preds
        if isinstance(pred_ids, tuple):
            pred_ids = pred_ids[0]

        # Newer transformers pads variable-length predictions across eval batches with -100;
        # replace those before decoding or the tokenizer raises an OverflowError.
        pred_ids = np.where(pred_ids != -100, pred_ids, processor.tokenizer.pad_token_id)

        # Replace ignore index to decode labels
        labels = np.where(label_ids != -100, label_ids, processor.tokenizer.pad_token_id)

        pred_texts = processor.batch_decode(pred_ids, skip_special_tokens=True)
        label_texts = processor.batch_decode(labels, skip_special_tokens=True)

        pred_texts = [p.strip() for p in pred_texts]
        label_texts = [l.strip() for l in label_texts]

        cer = _character_error_rate(label_texts, pred_texts)
        exact_match = sum(p == l for p, l in zip(pred_texts, label_texts)) / max(1, len(label_texts))
        return {"cer": cer, "exact_match": exact_match}

    return _compute_metrics


def build_training_args(output_dir: Path, use_fp16: bool) -> Seq2SeqTrainingArguments:
    """Build Seq2SeqTrainingArguments across transformers API variants."""
    common_kwargs = {
        "output_dir": str(output_dir),
        "per_device_train_batch_size": BATCH_SIZE,
        "per_device_eval_batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "num_train_epochs": EPOCHS,
        "save_strategy": "epoch",
        "logging_strategy": "steps",
        "logging_steps": 10,
        "predict_with_generate": True,
        "fp16": use_fp16,
        "save_total_limit": 2,
        "load_best_model_at_end": True,
        "metric_for_best_model": "cer",
        "greater_is_better": False,
        "report_to": "none",
    }

    params = inspect.signature(Seq2SeqTrainingArguments.__init__).parameters
    if "evaluation_strategy" in params:
        common_kwargs["evaluation_strategy"] = "epoch"
    elif "eval_strategy" in params:
        common_kwargs["eval_strategy"] = "epoch"
    else:
        logger.warning("No eval strategy arg found; falling back to no periodic evaluation.")

    return Seq2SeqTrainingArguments(**common_kwargs)


def build_trainer(
    model: VisionEncoderDecoderModel,
    training_args: Seq2SeqTrainingArguments,
    train_ds: Dataset,
    test_ds: Dataset,
    processor: TrOCRProcessor,
) -> Seq2SeqTrainer:
    """Build Seq2SeqTrainer across transformers API variants."""
    kwargs = {
        "model": model,
        "args": training_args,
        "train_dataset": train_ds,
        "eval_dataset": test_ds,
        "data_collator": TrOCRDataCollator(),
        "compute_metrics": build_compute_metrics(processor),
    }

    params = inspect.signature(Seq2SeqTrainer.__init__).parameters
    if "tokenizer" in params:
        kwargs["tokenizer"] = processor.tokenizer
    elif "processing_class" in params:
        kwargs["processing_class"] = processor
    else:
        logger.warning("No tokenizer/processing_class argument found in Seq2SeqTrainer.")

    return Seq2SeqTrainer(**kwargs)


# Checkpoints are written locally (fast disk); the final model is saved to Drive in Step 8.
CHECKPOINT_DIR = Path("/content/checkpoints")
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

use_fp16 = torch.cuda.is_available()
training_args = build_training_args(CHECKPOINT_DIR, use_fp16)
trainer = build_trainer(model, training_args, train_ds, test_ds, processor)

## Step 7 — Train

In [7]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 0}.


Epoch,Training Loss,Validation Loss,Cer,Exact Match
1,3.519658,3.784261,0.870168,0.000000
2,3.047488,3.565977,0.850000,0.000000
3,2.850361,3.283585,0.849580,0.006803
4,2.275272,3.180972,0.855252,0.000000
5,1.733041,3.077272,0.871218,0.000000
6,1.256979,3.159866,0.834034,0.000000
7,0.983154,3.165253,0.831513,0.006803
8,0.609295,3.267064,0.836134,0.006803
9,0.339098,3.310146,0.843067,0.000000
10,0.242424,3.346742,0.835504,0.006803


/usr/local/lib/python3.12/dist-packages/transformers/generation/utils.py:1612: UserWarning: Using the model-agnostic default `max_length` (=21) to control the generation length. We recommend setting `max_new_tokens` to control the maximum length of the generation.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['encoder.layers.0.attention.q_proj.weight', 'encoder.layers.0.attention.k_proj.weight', 'encoder.layers.0.attention.v_proj.weight', 'encoder.layers.0.attention.o_proj.weight', 'encoder.layers.0.attention.o_proj.bias', 'encoder.layers.0.layernorm_before.weight', 'encoder.layers.0.layernorm_before.bias', 'encoder.layers.0.layernorm_after.weight', 'encoder.layers.0.layernorm_after.bias', 'encoder.layers.0.mlp.fc1.weight', 'encoder.layers.0.mlp.fc1.bias', 'encoder.layers.0.mlp.fc2.weight', 'encoder.layers.0.mlp.fc2.bias', 'encoder.layers.1.attention.q_proj.weight', 'encoder.layers.1.attention.k_proj.weight', 'encoder.layers.1.attention.v_proj.weight', 'encoder.layers.1.attention.o_proj.weight', 'encoder.layers.1.attention.o_proj.bias', 'encoder.layers.1.layernorm_before.weight', 'encoder.layers.1.layernorm_before.bias', 'encoder.layers.1.layernorm_after.weight', 'encoder.layers.1.layernorm_after.bias', 'encoder.layers.

TrainOutput(global_step=700, training_loss=1.8364637122835432, metrics={'train_runtime': 4820.3775, 'train_samples_per_second': 1.728, 'train_steps_per_second': 0.145, 'total_flos': 6.233215640996413e+18, 'train_loss': 1.8364637122835432, 'epoch': 10.0})

In [8]:
eval_metrics = trainer.evaluate()
print("Final metrics:", eval_metrics)

Training Loss,Validation Loss,Epoch,Cer,Exact Match
0.242424,3.168569,10,0.832983,0.006803


Final metrics: {'eval_loss': 3.168569326400757, 'eval_cer': 0.832983193277311, 'eval_exact_match': 0.006802721088435374}


## Step 8 — Save the fine-tuned model to Drive

In [9]:
OUTPUT_MODEL_DIR.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(OUTPUT_MODEL_DIR))
processor.save_pretrained(str(OUTPUT_MODEL_DIR))
print("Saved model to", OUTPUT_MODEL_DIR)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved model to /content/drive/MyDrive/UMwTI/models/trocr_sample1000


## Step 9 — Qualitative test on held-out examples

In [10]:
def run_qualitative_test(
    model: VisionEncoderDecoderModel,
    processor: TrOCRProcessor,
    test_records: list[dict[str, str]],
    num_examples: int,
) -> None:
    """Print sample predictions from the test split."""
    if not test_records:
        print("No test samples available for qualitative test.")
        return

    model.eval()
    device = model.device

    subset = test_records[: min(num_examples, len(test_records))]
    for i, record in enumerate(subset, start=1):
        image = Image.open(record["image_path"]).convert("RGB")
        pixel_values = processor(images=image, return_tensors="pt").pixel_values.to(device)
        with torch.no_grad():
            generated_ids = model.generate(pixel_values, max_new_tokens=128)
        pred = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
        truth = record["text"].strip()
        print(f"[{i}] GT: {truth}")
        print(f"    PR: {pred}")


test_records = [test_raw[i] for i in range(len(test_raw))]
run_qualitative_test(trainer.model, processor, test_records, PRINT_TEST_EXAMPLES)

[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[1] GT: Wszystko u nas po staremu, zdrowie
    PR: chców wytrzyj, które wytrzej


[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[2] GT: zwraca uwagę, by pisać do niego
    PR: chciskam wytaję za bardzo,


[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


...

[7] GT: chczytaję zdrowia, zdrowia, które
    PR: Tak mały za bardzo, które


[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[8] GT: że mnie tak bardzo kochasz. Ja w to nie
    PR: chców wytrzyj, które wytrzyj


[transformers] Both `max_new_tokens` (=128) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[9] GT: miary Béhody do 8°. Nie quartę diz,
    PR: rzeczy zdrowia, i dziesię
[10] GT: Jego pomyślność najgorętsze moje życzenie.
    PR: Kochana Marysieńko, ktoże bymuntwo
